In [3]:
from nhlpy import NHLClient
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

In [4]:
PATH = "your_file.csv"   # .csv / .parquet / .xlsx
TARGET = None            # optional: name of the column you care about

if PATH.endswith(".csv"):
    df = pd.read_csv(PATH, low_memory=False)
elif PATH.endswith(".parquet"):
    df = pd.read_parquet(PATH)
else:
    df = pd.read_excel(PATH)

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'your_file.csv'

In [ ]:
# Try to parse object columns that look like dates
for c in df.select_dtypes(include="object"):
    parsed = pd.to_datetime(df[c], errors="coerce")
    if parsed.notna().mean() > 0.9:
        df[c] = parsed

dt_cols = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()

# Low-cardinality numerics (e.g. 0/1 flags, ratings) are often categorical in disguise
LOW_CARD = 10
disguised = [c for c in num_cols if df[c].nunique() <= LOW_CARD]
print("Numeric columns that may be categorical:", disguised)

summary = pd.DataFrame({
    "dtype": df.dtypes,
    "n_unique": df.nunique(),
    "pct_unique": df.nunique() / len(df) * 100,
    "n_missing": df.isna().sum(),
    "pct_missing": df.isna().mean() * 100,
    "example": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
})
summary

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum():,}")

constant = [c for c in df if df[c].nunique(dropna=False) <= 1]
print("Constant columns:", constant)

# Likely ID columns (nearly all unique) - usually useless for modelling
id_like = [c for c in df if df[c].nunique() / len(df) > 0.95 and c not in num_cols]
print("ID-like columns:", id_like)

# High-cardinality categoricals
high_card = [c for c in cat_cols if df[c].nunique() > 50]
print("High-cardinality categoricals:", high_card)

# Strings with stray whitespace / inconsistent casing
for c in cat_cols:
    s = df[c].dropna().astype(str)
    if (s != s.str.strip()).any():
        print(f"Whitespace issues in: {c}")
    if s.str.lower().nunique() < s.nunique():
        print(f"Case inconsistencies in: {c}")

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]

if len(miss):
    fig, axes = plt.subplots(1, 2, figsize=(14, max(3, len(miss) * 0.3)))
    miss.mul(100).plot.barh(ax=axes[0], color="tomato")
    axes[0].set(title="% missing by column", xlabel="%")
    axes[0].invert_yaxis()

    # Does missingness co-occur across columns?
    sns.heatmap(df[miss.index].isna().corr(), cmap="coolwarm", center=0,
                ax=axes[1], annot=len(miss) <= 12, fmt=".2f")
    axes[1].set_title("Missingness correlation")
    plt.tight_layout(); plt.show()
else:
    print("No missing values.")

## Montreal Canadians

In [ ]:
TEAM = "MTL"

In [ ]:
# Basic usage
client = NHLClient()

# Get all teams
mtl = client.teams.team_roster(team_abbr=f"{TEAM}", season="20252026")
display(mtl)
# Get current standings
standings = client.standings.league_standings()

# Get today's games
games = client.schedule.daily_schedule()

{'forwards': [{'id': 8476981,
   'headshot': 'https://assets.nhle.com/mugs/nhl/20252026/MTL/8476981.png',
   'firstName': {'default': 'Josh'},
   'lastName': {'default': 'Anderson'},
   'sweaterNumber': 17,
   'positionCode': 'R',
   'shootsCatches': 'R',
   'heightInInches': 75,
   'weightInPounds': 226,
   'heightInCentimeters': 191,
   'weightInKilograms': 103,
   'birthDate': '1994-05-07',
   'birthCity': {'default': 'Burlington'},
   'birthCountry': 'CAN',
   'birthStateProvince': {'default': 'ON'}},
  {'id': 8483424,
   'headshot': 'https://assets.nhle.com/mugs/nhl/20252026/MTL/8483424.png',
   'firstName': {'default': 'Owen'},
   'lastName': {'default': 'Beck'},
   'sweaterNumber': 62,
   'positionCode': 'C',
   'shootsCatches': 'R',
   'heightInInches': 72,
   'weightInPounds': 199,
   'heightInCentimeters': 183,
   'weightInKilograms': 90,
   'birthDate': '2004-02-03',
   'birthCity': {'default': 'Port Hope'},
   'birthCountry': 'CAN',
   'birthStateProvince': {'default': 'ON'

### Numeric features: distributions

In [ ]:

display(df[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T
        .assign(skew=df[num_cols].skew(), kurtosis=df[num_cols].kurt()))

def grid(n, ncols=3, h=3.2):
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, h * nrows))
    axes = np.array(axes).ravel()
    for ax in axes[n:]:
        ax.axis("off")
    return fig, axes

fig, axes = grid(len(num_cols))
for ax, c in zip(axes, num_cols):
    sns.histplot(df[c].dropna(), kde=True, ax=ax, bins=40)
    ax.set_title(f"{c}  (skew={df[c].skew():.2f})")
plt.tight_layout(); plt.show()

### Outliers

In [ ]:
def iqr_outliers(s, k=1.5):
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    return ((s < q1 - k * iqr) | (s > q3 + k * iqr)).sum()

out = pd.DataFrame({
    "n_outliers": {c: iqr_outliers(df[c].dropna()) for c in num_cols},
})
out["pct"] = out["n_outliers"] / len(df) * 100
display(out.sort_values("pct", ascending=False))

fig, axes = grid(len(num_cols), h=2.5)
for ax, c in zip(axes, num_cols):
    sns.boxplot(x=df[c], ax=ax, color="lightsteelblue")
    ax.set_title(c)
plt.tight_layout(); plt.show()

### Categorical Features

In [ ]:
for c in cat_cols:
    vc = df[c].value_counts(dropna=False)
    print(f"\n=== {c} ({df[c].nunique()} unique) ===")
    display(pd.DataFrame({"count": vc.head(10), "pct": (vc / len(df) * 100).head(10)}))

plot_cats = [c for c in cat_cols if 1 < df[c].nunique() <= 30]
if plot_cats:
    fig, axes = grid(len(plot_cats), ncols=2, h=3.5)
    for ax, c in zip(axes, plot_cats):
        order = df[c].value_counts().index[:15]
        sns.countplot(y=df[c], order=order, ax=ax, color="steelblue")
        ax.set_title(c)
    plt.tight_layout(); plt.show()

### Correlations

In [ ]:
if len(num_cols) > 1:
    corr = df[num_cols].corr(method="spearman")  # robust to skew and monotonic nonlinearity
    mask = np.triu(np.ones_like(corr, dtype=bool))
    plt.figure(figsize=(min(16, 0.6 * len(num_cols) + 4),) * 2)
    sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1,
                annot=len(num_cols) <= 15, fmt=".2f", square=True)
    plt.title("Spearman correlation"); plt.show()

    pairs = (corr.where(~mask).stack().rename("corr").reset_index()
             .rename(columns={"level_0": "a", "level_1": "b"}))
    pairs["abs"] = pairs["corr"].abs()
    print("Strongest pairs:")
    display(pairs.sort_values("abs", ascending=False).head(15).drop(columns="abs"))

### Categorical and numeric relationships

In [ ]:
# Cramér's V for categorical-categorical association
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return np.nan
    from scipy.stats import chi2_contingency
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    r, k = ct.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / max(1e-9, min(kc - 1, rc - 1)))

low_cat = [c for c in cat_cols if 1 < df[c].nunique() <= 20]
if len(low_cat) > 1:
    cv = pd.DataFrame(index=low_cat, columns=low_cat, dtype=float)
    for a in low_cat:
        for b in low_cat:
            cv.loc[a, b] = 1.0 if a == b else cramers_v(df[a], df[b])
    plt.figure(figsize=(8, 6))
    sns.heatmap(cv, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
    plt.title("Cramér's V"); plt.show()

# Numeric means/medians across categories: quick way to spot group effects
if low_cat and num_cols:
    c = low_cat[0]  # change to explore other groupings
    display(df.groupby(c)[num_cols].median().T)

### Target Analysis

In [ ]:
TARGET = f""

In [ ]:
if TARGET:
    y = df[TARGET]
    is_classification = (y.dtype == "object" or y.nunique() <= 10)
    feats_num = [c for c in num_cols if c != TARGET]
    print(f"Target treated as: {'classification' if is_classification else 'regression'}")

    if is_classification:
        print(y.value_counts(normalize=True).round(3))
        top = df[feats_num].apply(lambda s: s.corr(y.astype("category").cat.codes, method="spearman")) \
                           .abs().sort_values(ascending=False).head(9).index
        fig, axes = grid(len(top))
        for ax, c in zip(axes, top):
            sns.boxplot(data=df, x=TARGET, y=c, ax=ax)
            ax.set_title(c)
        plt.tight_layout(); plt.show()
    else:
        sns.histplot(y, kde=True); plt.title(f"Target: {TARGET}"); plt.show()
        cors = df[feats_num].corrwith(y, method="spearman").sort_values(key=abs, ascending=False)
        display(cors.head(15).to_frame("spearman_vs_target"))
        top = cors.head(6).index
        fig, axes = grid(len(top))
        for ax, c in zip(axes, top):
            sns.regplot(data=df.sample(min(3000, len(df))), x=c, y=TARGET, ax=ax,
                        scatter_kws={"alpha": .3, "s": 8}, lowess=True)
        plt.tight_layout(); plt.show()

#### Model-based signal check (scikit-learn)

In [ ]:
if TARGET:
    from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

    X = df.drop(columns=[TARGET] + dt_cols + id_like).copy()
    for c in X.select_dtypes(include=["object", "category", "bool", "string"]):
        X[c] = X[c].astype("category").cat.codes
    X = X.fillna(X.median(numeric_only=True))
    discrete = [X[c].nunique() <= LOW_CARD for c in X]

    yy = y.astype("category").cat.codes if is_classification else y.fillna(y.median())
    mi_fn = mutual_info_classif if is_classification else mutual_info_regression
    mi = pd.Series(mi_fn(X, yy, discrete_features=discrete, random_state=0), index=X.columns)
    mi.sort_values().tail(20).plot.barh(figsize=(7, 6), title="Mutual information with target")
    plt.show()

#### Time patterns

In [ ]:
for dc in dt_cols:
    print(f"{dc}: {df[dc].min()} -> {df[dc].max()}")
    ts = df.set_index(dc).sort_index()

    # Row volume over time
    ts.resample("D" if (df[dc].max() - df[dc].min()).days < 365 else "M").size() \
      .plot(figsize=(12, 3), title=f"Records over time ({dc})"); plt.show()

    # Trend of top numeric columns
    for c in num_cols[:4]:
        ts[c].resample("M").mean().plot(figsize=(12, 2.5), title=f"Monthly mean of {c}")
        plt.show()

    # Seasonality features
    tmp = df.assign(hour=df[dc].dt.hour, dow=df[dc].dt.dayofweek, month=df[dc].dt.month)
    if num_cols:
        fig, axes = plt.subplots(1, 2, figsize=(12, 3))
        tmp.groupby("dow")[num_cols[0]].mean().plot.bar(ax=axes[0], title=f"{num_cols[0]} by weekday")
        tmp.groupby("month")[num_cols[0]].mean().plot.bar(ax=axes[1], title=f"{num_cols[0]} by month")
        plt.tight_layout(); plt.show()

- Data quality: (missing %, duplicates, constants, IDs)
- Distributions: (skewed cols needing log transform, outliers to investigate)
- Relationships: (strong correlations, redundant features, group differences)
- Target signal: (top features, class imbalance)
- Time: (trends, seasonality, gaps)
- Next steps / questions for the data owner: